Foxf2iKO vs Ctrl differential expression (wilcoxon) across the full brain and per brain region, in all spots and in neuronal spots.
- Input: `geo/Visium_Foxf2iKO_Ctrl.h5ad` (from 03)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns
base = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/FOXF2/Visium")
for d in ("dea", "plots"):
    (base / d).mkdir(exist_ok=True)
adata = sc.read_h5ad(base / "geo" / "Visium_Foxf2iKO_Ctrl.h5ad")
counts = adata.X
adata.X = adata.layers["log1p_norm"]
adata.uns["log1p"] = {"base": None}
regions = ["Cortex", "Hippocampus", "FiberTract", "Thalamus", "Hypothalamus", "Striatum", "StriatumAmygdala", "Ventricle"]

In [ ]:
neuronal = ~adata.obs["neuronal_low"]
spots = {"all-spots": adata, "neuronal-spots": adata[neuronal]}
pd.crosstab(adata.obs["brain_region"], [adata.obs["genotype"], neuronal.rename("neuronal")])

In [ ]:
results = {}
for name, a in spots.items():
    res = {}
    for region in ["all"] + regions:
        sub = (a if region == "all" else a[a.obs["brain_region"] == region]).copy()
        if sub.obs["genotype"].value_counts().min() < 10:
            continue
        sc.tl.rank_genes_groups(sub, "genotype", groups=["Foxf2iKO"], reference="Ctrl", method="wilcoxon")
        df = sc.get.rank_genes_groups_df(sub, "Foxf2iKO").rename(columns={"names": "gene", "logfoldchanges": "log2FC", "scores": "zscore"})
        df.insert(2, "FC", 2 ** df["log2FC"])
        df["ensembl_id"] = df["gene"].map(adata.var["gene_ids"])
        res[region] = df.sort_values("pvals")
    results[name] = res
    with pd.ExcelWriter(base / "dea" / f"visium_Foxf2iKO-vs-Ctrl_wilcoxon_{name}.xlsx") as w:
        for region, df in res.items():
            df.to_excel(w, sheet_name=region, index=False)

pd.DataFrame({name: {r: (df["pvals_adj"] < 0.05).sum() for r, df in res.items()} for name, res in results.items()})

DEGs per brain region (all spots, FDR < 0.05), as % of genes detected in the region

In [ ]:
res = results["all-spots"]
metrics = adata.obs.groupby("brain_region", observed=True).agg(n_spots=("sample", "size"), n_genes_mean=("n_genes", "mean"), n_counts_mean=("total_counts", "mean"))
metrics = metrics.loc[regions]
metrics["n_up"] = [((res[r]["pvals_adj"] < 0.05) & (res[r]["log2FC"] > 0)).sum() for r in regions]
metrics["n_down"] = [((res[r]["pvals_adj"] < 0.05) & (res[r]["log2FC"] < 0)).sum() for r in regions]
metrics["n_genes_detected"] = [(counts[(adata.obs["brain_region"] == r).values].sum(0) > 0).sum() for r in regions]
metrics["pct_deg"] = 100 * (metrics["n_up"] + metrics["n_down"]) / metrics["n_genes_detected"]
metrics.to_csv(base / "dea" / "visium_Foxf2iKO-vs-Ctrl_wilcoxon_all-spots_region-metrics.csv")
metrics.corr()["pct_deg"]  # DEG numbers vs region size and depth

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for thr, up, down in [(0.1, "#ff9999", "#b3d1ff"), (0.3, "#e60000", "#002db3")]:
    ax.barh(regions, [((res[r]["pvals_adj"] < 0.05) & (res[r]["log2FC"] > thr)).sum() for r in regions], color=up, label=f"up, log2FC > {thr}")
    ax.barh(regions, [-((res[r]["pvals_adj"] < 0.05) & (res[r]["log2FC"] < -thr)).sum() for r in regions], color=down, label=f"down, log2FC < -{thr}")
ax.invert_yaxis()
ax.set_xlabel("DEGs Foxf2iKO vs Ctrl (FDR < 0.05)")
ax.legend(frameon=False, fontsize=8)
plt.show()

Foxf2 per region: baseline expression (Ctrl) and Foxf2iKO vs Ctrl

In [ ]:
sc.pl.matrixplot(adata[adata.obs["genotype"] == "Ctrl"], ["Foxf2"], groupby="brain_region", standard_scale="var", figsize=(2, 5))
pd.DataFrame({r: res[r].set_index("gene").loc["Foxf2", ["log2FC", "pvals", "pvals_adj"]] for r in res}).T

Ferroptosis genes: neuronal spots and all KEGG ferroptosis genes (mmu04216) per region, all spots

In [ ]:
genes = ["Ftl1", "Fth1", "Tfrc", "Trf", "Gpx4", "Vdac2", "Prnp", "Acsl6", "Acsl4", "Pcbp1", "Slc7a11"]
kegg = ["Slc3a2", "Slc7a11", "Trp53", "Gclc", "Gclm", "Gss", "Gpx4", "Acsl1", "Acsl3", "Acsl4", "Acsl5", "Acsl6", "Lpcat3", "Alox15", "Sat1", "Sat2",
        "Trf", "Tfrc", "Steap3", "Slc11a2", "Slc39a8", "Slc39a14", "Pcbp1", "Pcbp2", "Slc40a1", "Cp", "Fth1", "Ftl1", "Map1lc3a", "Map1lc3b",
        "Atg5", "Atg7", "Ncoa4", "Prnp", "Hmox1", "Vdac2", "Vdac3", "Cybb", "Ftmt"]

for name, gs, cols, size in [("neuronal-spots", genes, ["all", "Cortex", "Hippocampus"], (2.5, 5)), ("all-spots", kegg, ["all"] + regions, (5, 11))]:
    res = results[name]
    fc = pd.DataFrame({r: res[r].set_index("gene")["FC"] for r in cols}).reindex(gs).dropna(how="all")
    fdr = pd.DataFrame({r: res[r].set_index("gene")["pvals_adj"] for r in cols}).reindex(fc.index)
    plt.figure(figsize=size)
    sns.heatmap(fc, center=1, vmin=0.75, vmax=1.3, cmap="RdBu_r", annot=fdr.map(lambda p: f"{p:.1e}"), fmt="", annot_kws={"size": 6}, cbar_kws={"label": "FC (Foxf2iKO vs Ctrl)"})
    plt.title(f"{name}, values: FDR")
    plt.savefig(base / "plots" / f"heatmap_ferroptosis_{name}.svg", bbox_inches="tight")
    plt.show()